In [38]:
import time
import pickle
import numpy as np
import scipy
from scipy.linalg import sqrtm, inv
from sklearn.covariance import GraphicalLassoCV
from tqdm import tqdm
from functions.gram_matrix import gram_matrix2, gram_matrix
from functions.lib_fun import gwire_cv
from functions.FOPG import FOPG
from functions.FD_SDR import FD_SDR

def smvnormal(d, M=None, Sigma=None):
    """Symmetric matrix variate normal distribution."""
    if M is None:
        M = np.zeros((d, d))
    if Sigma is None:
        Sigma = np.eye(d)
    if np.linalg.norm(M - M.T, 'fro') > 1e-8:
        raise ValueError('Mean covariate must be symmetric')
    if np.any(np.linalg.eigvals(Sigma) < 0):
        raise ValueError('Covariance matrix must be symmetric positive definite')
    
    x = np.zeros((d, d))
    x[np.diag_indices(d)] = np.random.randn(d)
    indices = np.tril_indices(d, -1)
    x[indices] = np.random.randn(len(indices[0])) / np.sqrt(2)
    x = x + x.T - np.diag(np.diag(x))
    G = sqrtm(Sigma)
    return G.T @ x @ G + M

def direction_error_angle(b, beta_true):
    """Calculate the directional error between estimated and true coefficients."""
    proj_b = b @ inv(b.T @ b) @ b.T
    proj_true = beta_true @ inv(beta_true.T @ beta_true) @ beta_true.T
    return np.linalg.norm(proj_b - proj_true, 'fro')

def generate_X(n, p, mode_X = '(a)'):
    """Data generation for X"""
    if mode_X == '(a)':
        X = np.random.randn(n, p)
    elif mode_X == '(b)':
        X = np.zeros((n, p))
        phi = 0.5
        sigma_epsilon = np.sqrt(1 - phi**2)
        # Generate X
        for sample in range(n):
            U = np.zeros(p)
            U[0] = np.random.normal(0, 1)
        
            for t in range(1, p):
                epsilon_t = np.random.normal(0, sigma_epsilon)
                U[t] = phi * U[t-1] + epsilon_t
        
            # Modify U[0] and U[1]
            U[0] = np.sin(U[0])
            U[1] = np.abs(U[1])
        
            X[sample, :] = U

    elif mode_X == '(c)':
        X = np.zeros((n, p))
        phi = 0.5
        sigma_epsilon = np.sqrt(1 - phi**2)
        for sample in range(n):
            U = np.zeros(p)
            U[0] = np.random.normal(0, 1)

            for t in range(1, p):
                epsilon_t = np.random.normal(0, sigma_epsilon)
                U[t] = phi * U[t-1] + epsilon_t

            x = scipy.stats.norm.cdf(U)
            X[sample, :] = x

    return X

def generate_Y(X, alpha = 0.2, q = 100, mode_y = '(1)', eps = 0.5, IF_GWIRE = False, neigh = None):
    n = X.shape[0]
    p = X.shape[1]

    beta_1 = np.concatenate([np.array([1, 1]), np.zeros(p - 2)])
    beta_2 = np.concatenate([np.zeros(p - 2), np.array([1, 1])])
    beta_3 = np.concatenate([[1, 2], np.zeros(p - 3), [2]])
    beta_4 = np.concatenate(([0, 0, 1, 2, 2], np.zeros(p - 5)))
    
    """Neighborhood estimation for GWIRE"""
    Nb = []
    if IF_GWIRE:
        if not neigh:
            # Neighborhood is unknown
            graphical_lasso_model = GraphicalLassoCV()
            graphical_lasso_model.fit(X)
            omega = graphical_lasso_model.precision_
            np.where(np.sum(omega!=0, axis = 1) > 1)[0]
            
        for j in range(p):
            Ni = (np.nonzero(omega[j, :])[0]).tolist()
            Nb.append(Ni)

    """Data generation for Y"""
    if mode_y == '(1)':
        # generate beta_true
        d_0 = 1
        d = 2
        beta_true = beta_1.reshape((p,1))
        
        DX = [np.array([[1, 0.8*np.cos(np.dot(x, beta_1))],
                       [0.8*np.cos(np.dot(x, beta_1)), 1]]) for x in X]
        
    elif mode_y == '(2)':
        # generate beta_true
        d_0 = 2
        d = 3
        beta_true = np.vstack([beta_1,beta_2]).T

        DX = [np.array([[1, 0.8*np.cos(np.dot(x, beta_1)),0.8*np.sin(np.dot(x, beta_2))],
                        [0.8*np.cos(np.dot(x, beta_1)), 1, 0.8*np.cos(np.dot(x, beta_2))],
                        [0.8*np.sin(np.dot(x, beta_2)), 0.8*np.cos(np.dot(x, beta_2)), 1]]) for x in X]

    def geny(dx):
        return np.round((scipy.linalg.expm(smvnormal(d, scipy.linalg.logm(dx), eps * np.eye(d))) +
                        scipy.linalg.expm(smvnormal(d, scipy.linalg.logm(dx), eps * np.eye(d)))) / 2, 7)
    
    Y = np.array([geny(dx) for dx in DX])
    return {'X': X, 'y': Y, 'beta_true': beta_true, 'Nb': Nb, 'd_0': d_0}

def run_simulation(config):
    num_repeats = config['num_repeats']
    n = config['n']
    p = config['p']
    q = config['q']
    alpha = config['alpha']
    mode_X = config['mode_X']
    mode_y = config['mode_y']
    IF_GWIRE = config['IF_GWIRE']
    neigh = config['neigh']
    verbose = config['verbose']
    metric = config['metric']
    
    # Initialize result storage
    results = {
        'gwire_times': [],
        'fopg_times': [],
        'fd_sdr_times': [],
        'gwire_errors': [],
        'fd_sdr_errors': [],
        'fopg_errors': []
    }

    """Run simulation"""
    if verbose:
        print("\n" + "="*60)
        print(f"Starting Simulation".center(60))
        print("="*60)
        print(f"Configuration:")
        print(f"- Repeats: {num_repeats}")
        print(f"- Dimensions: n={n}, p={p}, q={q}")
        print(f"- X mode: {mode_X}, Y mode: {mode_y}")
        print(f"- Metric: {metric}")
        print(f"- GWIRE: {'Enabled' if IF_GWIRE else 'Disabled'}")
        print("="*60 + "\n")

    # Use tqdm for progress bar if verbose is False
    iterator = range(num_repeats)
    if not verbose:
        print(f"Parameters: n = {n}, p = {p}, q = {q}, mode_X = {mode_X}, mode_y = {mode_y}, alpha = {alpha}, IF_GWIRE = {IF_GWIRE}")
        iterator = tqdm(iterator, desc="Running simulation", unit="iter")


    for i in iterator:
        iter_start = time.time()
        if verbose:
            print(f"\nIteration {i+1}/{num_repeats} ".ljust(30, '-'))

        X = generate_X(n, p, mode_X)
        DATA_XY = generate_Y(X, alpha, q, mode_y, IF_GWIRE, neigh)
        y = DATA_XY['y']
        beta_true = DATA_XY['beta_true']
        Nb = DATA_XY['Nb']
        d_0 = DATA_XY['d_0']
        
        iter_start = time.time()
        # ygram = sqrtm(gram_matrix2(y, 10))
        # ygram2 = gram_matrix2(y, 1)
        ygram = gram_matrix(y, 1000)
        ygram2 = gram_matrix(y, 1)

        # GWIRE method
        if IF_GWIRE:
            start_time = time.time()
            beta_gwire, _ = gwire_cv(X, y, Nb, metric, d_0, fold=5)
            gwire_time = time.time() - start_time
            results['gwire_times'].append(gwire_time)
            gwire_error = direction_error_angle(beta_gwire, beta_true)
            results['gwire_errors'].append(gwire_error)
            if verbose:
                print(f"GWIRE: Time={gwire_time:.3f}s, Error={gwire_error:.4f}")
        
        # FOPG method
        start_time = time.time()
        beta_fopg = FOPG(X, ygram2, d_0)
        fopg_time = time.time() - start_time
        results['fopg_times'].append(fopg_time)
        fopg_error = direction_error_angle(beta_fopg, beta_true)
        results['fopg_errors'].append(fopg_error)
        if verbose:
            print(f"FOPG:  Time={fopg_time:.3f}s, Error={fopg_error:.4f}")

        # FD-SDR method
        start_time = time.time()
        beta_fd_sdr, _, _ = FD_SDR(X.T, ygram, beta_fopg)
        fd_sdr_time = time.time() - start_time
        results['fd_sdr_times'].append(fd_sdr_time)
        fd_sdr_error = direction_error_angle(beta_fd_sdr, beta_true)
        results['fd_sdr_errors'].append(fd_sdr_error)
        if verbose:
            print(f"FD-SDR: Time={fd_sdr_time:.3f}s, Error={fd_sdr_error:.4f}")

        # Print iteration summary
        iter_time = time.time() - iter_start
        if verbose:
            print("-"*40)
            print(f"Iteration completed in {iter_time:.2f} seconds")
            print("-"*40)

    return results

def print_summary(results, config=None):
    """
    Print a well-formatted summary of the simulation results.
    
    Parameters:
    -----------
    results : dict
        Dictionary containing the simulation results with keys:
        - 'gwire_times', 'fopg_times', 'fd_sdr_times' (lists of times)
        - 'gwire_errors', 'fopg_errors', 'fd_sdr_errors' (lists of errors)
        
    config : dict, optional
        Dictionary containing simulation configuration parameters
    """
    # Header
    print("\n" + "="*80)
    print(" SIMULATION SUMMARY ".center(80, '='))
    print("="*80)
    
    # Print configuration if provided
    if config:
        print("\nCONFIGURATION:")
        for key, value in config.items():
            print(f"- {key}: {value}")
        print("-"*80)
    
    # Calculate statistics
    stats = {}
    methods = []
    
    if 'gwire_times' in results and len(results['gwire_times']) > 0:
        methods.append('GWIRE')
        stats['GWIRE'] = {
            'time_mean': np.mean(results['gwire_times']),
            'time_std': np.std(results['gwire_times']),
            'error_mean': np.mean(results['gwire_errors']),
            'error_std': np.std(results['gwire_errors'])
        }
    
    methods.extend(['FOPG', 'FD-SDR'])
    
    stats['FOPG'] = {
        'time_mean': np.mean(results['fopg_times']),
        'time_std': np.std(results['fopg_times']),
        'error_mean': np.mean(results['fopg_errors']),
        'error_std': np.std(results['fopg_errors'])
    }
    
    stats['FD-SDR'] = {
        'time_mean': np.mean(results['fd_sdr_times']),
        'time_std': np.std(results['fd_sdr_times']),
        'error_mean': np.mean(results['fd_sdr_errors']),
        'error_std': np.std(results['fd_sdr_errors'])
    }
    
    # Print performance table
    print("\nPERFORMANCE METRICS:")
    print("-"*80)
    print(f"{'Method':<10}{'Time (mean ± std)':<30}{'Error (mean ± std)':<30}")
    print("-"*80)
    
    for method in methods:
        m = stats[method]
        time_str = f"{m['time_mean']:.4f}s ± {m['time_std']:.4f}"
        error_str = f"{m['error_mean']:.4f} ± {m['error_std']:.4f}"
        print(f"{method:<10}{time_str:<30}{error_str:<30}")
    
    # Footer
    print("="*80 + "\n")


In [8]:
import pickle
import os
from datetime import datetime
from glob import glob

def save_results(results, config=None, result_name="simulation_results", directory='results'):
    """
    Save simulation results to [result_name].pkl file
    
    Parameters:
    -----------
    results : dict
        Dictionary containing simulation results
    config : dict, optional
        Dictionary containing simulation configuration  
    result_name : str
        Base name for the results file (without extension)
        Default: "simulation_results"
    directory : str, optional
        Directory to save results (default: 'results')
    
    Returns:
    --------
    str
        Full path to the saved .pkl file
        
    Example:
    --------
    >>> save_results(results, config, "A")
    'results/A.pkl'
    """
    # Create directory if it doesn't exist
    os.makedirs(directory, exist_ok=True)
    
    # Create filename with .pkl extension
    filename = f"{result_name}.pkl"
    filepath = os.path.join(directory, filename)
    
    # Prepare data to save
    data = {
        'results': results,
        'config': config,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    # Save to pickle file
    with open(filepath, 'wb') as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    print(f"Results saved to: {filepath}")
    # return filepath

def load_single_result(result_name="simulation_results", directory='results'):
    """
    Load simulation results from [result_name].pkl file
    
    Parameters:
    -----------
    result_name : str
        Base name for the results file (without extension)
        Default: "simulation_results"
    directory : str, optional
        Directory where results are saved (default: 'results')
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'results': the simulation results
        - 'config': the simulation configuration (if available)
        - 'timestamp': when the results were saved
        
    Raises:
    -------
    FileNotFoundError
        If the specified results file doesn't exist
        
    Example:
    --------
    >>> data = load_results("A")
    >>> results = data['results']
    >>> config = data['config']
    """
    # Create filename with .pkl extension
    filename = f"{result_name}.pkl"
    filepath = os.path.join(directory, filename)
    
    # Check if file exists
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"No results file found at: {filepath}")
    
    # Load data from pickle file
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    # print(f"Results loaded from: {filepath}")
    return data

def load_all_results(directory='results'):
    """
    Load all results from the specified directory
    
    Parameters:
    -----------
    directory : str, optional
        Directory where results are saved (default: 'results')
    
    Returns:
    --------
    list
        List of dictionaries containing results and configurations
    """
    filepaths = sorted(glob(os.path.join(directory, '*.pkl')))
    results_list = []
    configs_list = []

    for filepath in filepaths:
        # Get filename without extension
        filename = os.path.splitext(os.path.basename(filepath))[0]
        data = load_single_result(filename, directory)
        results_list.append(data['results'])
        configs_list.append(data['config'])

    return results_list, configs_list

In [9]:
from tabulate import tabulate
import numpy as np

def results_to_table(results_list, configs_list, IF_TIME=False):
    """
    Convert multiple simulation results into a single formatted table.
    
    Parameters:
    -----------
    results_list : list of dict
        List of result dictionaries, each in format:
        {
            'gwire_errors': [0.1, 0.2, 0.3],    
            'fopg_errors': [0.2, 0.3, 0.4],
            'fd_sdr_errors': [0.3, 0.4, 0.5],
            'gwire_times': [0.1, 0.2, 0.3],
            'fopg_times': [0.2, 0.3, 0.4],
            'fd_sdr_times': [0.3, 0.4, 0.5]
        }
    configs_list : list of dict
        List of configuration dictionaries corresponding to each result
    
    Returns:        
    --------
    str
        Formatted table as string
    """
    # Helper function to handle empty lists
    def format_error(errors):
        if not errors:  # if list is empty
            return "N/A"
        return f"{np.mean(errors):.2f}({np.std(errors):.2f})"

    # Helper function to get model description
    def get_model_description(config):
        mode_X = config['mode_X']
        mode_y = config['mode_y']
        model = mode_y[:2] + ' - ' + mode_X[1:]
        if mode_y in ['(3)', '(4)']:
            model += f', α={config["alpha"]}'
        return model

    # Table headers
    headers = ["(n,p)", "Model", "Fd-SDR", "FOPG", "GWIRE"]
    
    table_data = []
    for result, config in zip(results_list, configs_list):
        n = config['n']
        p = config['p']
        model = get_model_description(config)
        
        if IF_TIME:
            row = [
                f"({n},{p})",
                model,
                format_error(result.get('fd_sdr_times', [])),
                format_error(result.get('fopg_times', [])),
                format_error(result.get('gwire_times', []))
            ]
        else:
            row = [
                f"({n},{p})",
                model,
                format_error(result.get('fd_sdr_errors', [])),
                format_error(result.get('fopg_errors', [])),
                format_error(result.get('gwire_errors', []))
            ]
        table_data.append(row)
    
    print(tabulate(table_data, headers=headers, tablefmt="grid", stralign="center"))
    # return table_data

# Example usage:
# results_to_table([results_200_10], [config])

In [33]:
# Single simulation
np.random.seed(123)

config = {
    'n': 200,
    'p': 10,
    'q': 100,
    'alpha': 0.2,
    'mode_X': '(a)',
    'mode_y': '(1)',
    'num_repeats': 1,
    'IF_GWIRE': False,
    'neigh': None,
    'metric': 'Frobenius',
    'verbose': True
}

results = run_simulation(config = config)
result_name = f"results-{config['n']}-{config['p']}-{config['mode_X']}-{config['mode_y']}"
save_results(results, config, result_name=result_name, directory='results-II')


                    Starting Simulation                     
Configuration:
- Repeats: 10
- Dimensions: n=200, p=10, q=100
- X mode: (a), Y mode: (1)
- Metric: Frobenius
- GWIRE: Disabled


Iteration 1/10 --------------
FOPG:  Time=1.519s, Error=0.0479
FD-SDR: Time=0.206s, Error=0.0723
----------------------------------------
Iteration completed in 1.77 seconds
----------------------------------------

Iteration 2/10 --------------
FOPG:  Time=1.640s, Error=0.0839
FD-SDR: Time=0.118s, Error=0.0493
----------------------------------------
Iteration completed in 1.82 seconds
----------------------------------------

Iteration 3/10 --------------
FOPG:  Time=1.541s, Error=0.1063
FD-SDR: Time=0.083s, Error=0.0642
----------------------------------------
Iteration completed in 1.69 seconds
----------------------------------------

Iteration 4/10 --------------
FOPG:  Time=1.459s, Error=0.1612
FD-SDR: Time=0.078s, Error=0.0820
----------------------------------------
Iteration completed in 

In [ ]:
# Define the parameter combinations to test
# n_p_combinations = [(200, 10), (400, 20), (600, 100)]
n_p_combinations = [(600, 100)]

mode_combinations = [
    ('(2)', '(a)'), ('(2)', '(b)')
]

# mode_combinations = [
#     ('(1)', '(a)'), ('(1)', '(b)'),
#     ('(2)', '(a)'), ('(2)', '(b)')
# ]

# Base configuration
base_config = {
    'num_repeats': 100,
    'q': 100,
    'alpha': 1,
    'neigh': None,
    'metric': 'Wasserstein',
    'verbose': False
}

# Run simulations for all combinations
for n, p in n_p_combinations:
    if p > 20:
        IF_GWIRE = False
    else:
        IF_GWIRE = True
        
    for mode_y, mode_X in mode_combinations:
        config = base_config.copy()
        config.update({
            'n': n,
            'p': p,
            'mode_X': mode_X,
            'mode_y': mode_y,
            'IF_GWIRE': IF_GWIRE
        })
        
        results = run_simulation(config=config)
        result_name = f"results-{n}-{p}-{mode_y}-{mode_X}"
        save_results(results, config, result_name=result_name, directory='results-II')
        print(f"Completed: n={n}, p={p}, modes=({mode_X},{mode_y})")

Parameters: n = 600, p = 100, q = 100, mode_X = (a), mode_y = (2), alpha = 1, IF_GWIRE = False


Running simulation:   0%|          | 0/2 [00:00<?, ?iter/s]/Users/feng/Library/CloudStorage/OneDrive-TheUniversityofTexasatElPaso/Research/2-In Progress/5-Frechet SDR/code/functions/gram_matrix.py:23: ComplexWarning: Casting complex values to real discards the imaginary part
  X = np.array(X, dtype=float)
Running simulation:   0%|          | 0/2 [00:08<?, ?iter/s]


KeyboardInterrupt: 

In [21]:
results_list, configs_list = load_all_results(directory='results-II')

# Results Table for Errors
results_to_table(results_list, configs_list, IF_TIME=False)

+----------+---------+------------+------------+---------+
|  (n,p)   |  Model  |   Fd-SDR   |    FOPG    |  GWIRE  |
+==========+=========+============+============+=========+
| (200,10) | (1 - a) | 0.06(0.01) | 0.11(0.05) |   N/A   |
+----------+---------+------------+------------+---------+
| (200,10) | (1 - b) | 0.16(0.00) | 0.06(0.00) |   N/A   |
+----------+---------+------------+------------+---------+


In [22]:
# Results Table for Runtime
results_to_table(results_list, configs_list, IF_TIME=True)

+----------+---------+------------+------------+---------+
|  (n,p)   |  Model  |   Fd-SDR   |    FOPG    |  GWIRE  |
+==========+=========+============+============+=========+
| (200,10) | (1 - a) | 0.07(0.01) | 1.53(0.06) |   N/A   |
+----------+---------+------------+------------+---------+
| (200,10) | (1 - b) | 0.09(0.00) | 1.62(0.00) |   N/A   |
+----------+---------+------------+------------+---------+
